In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import seaborn as sns

from scipy.spatial import distance

from sklearn.model_selection import train_test_split, StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler, MinMaxScaler

In [2]:
df = pd.read_csv("C:/Users/Asus/montesinho-fire-risk-prediction/data/raw/forestfires.csv")

display(df.describe().T)
display(df.head(10))
display(df.info())

,count,mean,std,min,25%,50%,75%,max
X,517.0,4.669246,2.313778,1.0,3.0,4.00,7.00,9.00
Y,517.0,4.299807,1.229900,2.0,4.0,4.00,5.00,9.00
FFMC,517.0,90.644681,5.520111,18.7,90.2,91.60,92.90,96.20
DMC,517.0,110.872340,64.046482,1.1,68.6,108.30,142.40,291.30
DC,517.0,547.940039,248.066192,7.9,437.7,664.20,713.90,860.60
ISI,517.0,9.021663,4.559477,0.0,6.5,8.40,10.80,56.10
temp,517.0,18.889168,5.806625,2.2,15.5,19.30,22.80,33.30
RH,517.0,44.288201,16.317469,15.0,33.0,42.00,53.00,100.00
wind,517.0,4.017602,1.791653,0.4,2.7,4.00,4.90,9.40
rain,517.0,0.021663,0.295959,0.0,0.0,0.00,0.00,6.40


,X,Y,month,day,FFMC,DMC,DC,ISI,temp,RH,wind,rain,area
0,7,5,mar,fri,86.2,26.2,94.3,5.1,8.2,51,6.7,0.0,0.0
1,7,4,oct,tue,90.6,35.4,669.1,6.7,18.0,33,0.9,0.0,0.0
2,7,4,oct,sat,90.6,43.7,686.9,6.7,14.6,33,1.3,0.0,0.0
3,8,6,mar,fri,91.7,33.3,77.5,9.0,8.3,97,4.0,0.2,0.0
4,8,6,mar,sun,89.3,51.3,102.2,9.6,11.4,99,1.8,0.0,0.0
5,8,6,aug,sun,92.3,85.3,488.0,14.7,22.2,29,5.4,0.0,0.0
6,8,6,aug,mon,92.3,88.9,495.6,8.5,24.1,27,3.1,0.0,0.0
7,8,6,aug,mon,91.5,145.4,608.2,10.7,8.0,86,2.2,0.0,0.0
8,8,6,sep,tue,91.0,129.5,692.6,7.0,13.1,63,5.4,0.0,0.0
9,7,5,sep,sat,92.5,88.0,698.6,7.1,22.8,40,4.0,0.0,0.0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 517 entries, 0 to 516
Data columns (total 13 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   X       517 non-null    int64  
 1   Y       517 non-null    int64  
 2   month   517 non-null    object 
 3   day     517 non-null    object 
 4   FFMC    517 non-null    float64
 5   DMC     517 non-null    float64
 6   DC      517 non-null    float64
 7   ISI     517 non-null    float64
 8   temp    517 non-null    float64
 9   RH      517 non-null    int64  
 10  wind    517 non-null    float64
 11  rain    517 non-null    float64
 12  area    517 non-null    float64
dtypes: float64(8), int64(3), object(2)
memory usage: 52.6+ KB


None

In [3]:
df_fe = df.copy()

month_map = {'jan': 1, 'feb': 2, 'mar': 3, 'apr': 4, 'may': 5, 'jun': 6, 
             'jul': 7, 'aug': 8, 'sep': 9, 'oct': 10, 'nov': 11, 'dec': 12}
day_map = {'mon': 1, 'tue': 2, 'wed': 3, 'thu': 4, 'fri': 5, 'sat': 6, 'sun': 7}

df_fe['month_num'] = df_fe['month'].map(month_map)
df_fe['day_num'] = df_fe['day'].map(day_map)

print(df_fe[['month', 'month_num', 'day', 'day_num']].head(3))

  month  month_num  day  day_num
0   mar          3  fri        5
1   oct         10  tue        2
2   oct         10  sat        6


In [4]:
from scipy.spatial import distance

condition_cols = ['month', 'day', 'temp', 'RH', 'wind', 'rain', 'FFMC', 'DMC', 'DC', 'ISI']

df_fe['same_condition_count'] = df_fe.groupby(condition_cols)['area'].transform('count')

df_fe['daily_total_area'] = df_fe.groupby(condition_cols)['area'].transform('sum')

df_fe['ignition_type'] = 0 

multi_fire_groups = df_fe[df_fe['same_condition_count'] > 1].groupby(condition_cols)

for name, group in multi_fire_groups:
    coords = group[['X', 'Y']].values
    wind_speed = group['wind'].iloc[0] 
    max_dist = 0
    
    if len(coords) > 1:
        dist_matrix = distance.cdist(coords, coords, 'euclidean')
        max_dist = np.max(dist_matrix)
    
    if max_dist >= 2.5:
        df_fe.loc[group.index, 'ignition_type'] = 2
        
    elif max_dist < 2.5 and (3.5 <= wind_speed <= 6.0):
        df_fe.loc[group.index, 'ignition_type'] = 1
        
    else:
        df_fe.loc[group.index, 'ignition_type'] = 2

print(df_fe[df_fe['same_condition_count'] > 1][['X', 'Y', 'wind', 'same_condition_count', 'ignition_type']].head(5))

    X  Y  wind  same_condition_count  ignition_type
14  6  5   4.5                     2              1
26  7  4   5.8                     2              1
49  4  4   5.8                     3              2
52  4  3   4.9                     2              1
53  4  3   4.9                     2              1


In [5]:
df_fe['distance_to_center'] = np.sqrt((df_fe['X'] - 4)**2 + (df_fe['Y'] - 4)**2)

df_fe['is_periphery'] = ((df_fe['X'] == 1) | (df_fe['X'] == 9) | 
                         (df_fe['Y'] == 1) | (df_fe['Y'] == 9)).astype(int)

df_fe['human_access_zone'] = ((df_fe['X'] == 8) & (df_fe['Y'] == 6)).astype(int)

def calculate_dynamic_distance(row):
    
    if row['month_num'] in [6, 7, 8, 9]:
        target_x, target_y = 8, 6
    else:
        target_x, target_y = 6, 5
        
    return np.sqrt((row['X'] - target_x)**2 + (row['Y'] - target_y)**2)

df_fe['dynamic_seasonal_hotspot_distance'] = df_fe.apply(calculate_dynamic_distance, axis=1)

print(df_fe[['X', 'Y', 'month_num', 'distance_to_center', 'dynamic_seasonal_hotspot_distance']].head())

   X  Y  month_num  distance_to_center  dynamic_seasonal_hotspot_distance
0  7  5          3            3.162278                           1.000000
1  7  4         10            3.000000                           1.414214
2  7  4         10            3.000000                           1.414214
3  8  6          3            4.472136                           2.236068
4  8  6          3            4.472136                           2.236068


In [6]:
def get_day_type(day_num):
    if day_num == 6:
        return 'Saturday' 
    elif day_num == 7:
        return 'Sunday'  
    else:
        return 'Weekday' 

df_fe['day_type'] = df_fe['day_num'].apply(get_day_type)

df_fe = pd.get_dummies(df_fe, columns=['day_type'], drop_first=False)

for col in ['day_type_Saturday', 'day_type_Sunday', 'day_type_Weekday']:
    if col in df_fe.columns:
        df_fe[col] = df_fe[col].astype(int)

peak_months = [3, 5, 8, 9, 12]
df_fe['is_peak_season'] = df_fe['month_num'].isin(peak_months).astype(int)

df_fe['month_sin'] = np.sin(2 * np.pi * df_fe['month_num'] / 12.0)
df_fe['month_cos'] = np.cos(2 * np.pi * df_fe['month_num'] / 12.0)

df_fe['day_sin'] = np.sin(2 * np.pi * df_fe['day_num'] / 7.0)
df_fe['day_cos'] = np.cos(2 * np.pi * df_fe['day_num'] / 7.0)

print(df_fe[['month_num', 'is_peak_season', 'day_type_Saturday', 'day_type_Sunday', 'month_sin']].head())

   month_num  is_peak_season  day_type_Saturday  day_type_Sunday  month_sin
0          3               1                  0                0   1.000000
1         10               0                  0                0  -0.866025
2         10               0                  1                0  -0.866025
3          3               1                  0                0   1.000000
4          3               1                  0                1   1.000000


In [7]:
df_fe['moderate_wind_danger'] = ((df_fe['wind'] >= 3.5) & (df_fe['wind'] <= 6.0)).astype(int)

df_fe['has_rain'] = (df_fe['rain'] > 0).astype(int)

df_fe['hot_and_dry'] = ((df_fe['temp'] >= 20.0) & (df_fe['RH'] <= 40.0)).astype(int)

df_fe['double_drought'] = ((df_fe['FFMC'] >= 88.0) & (df_fe['DC'] >= 500.0)).astype(int)

df_fe['ISI_x_DC'] = df_fe['ISI'] * df_fe['DC']

df_fe['rain_evaporation_risk'] = ((df_fe['rain'] > 0) & (df_fe['temp'] > 21.0)).astype(int)

print(df_fe[['temp', 'RH', 'wind', 'rain', 'hot_and_dry', 'moderate_wind_danger', 'rain_evaporation_risk']].head())

   temp  RH  wind  rain  hot_and_dry  moderate_wind_danger  \
0   8.2  51   6.7   0.0            0                     0   
1  18.0  33   0.9   0.0            0                     0   
2  14.6  33   1.3   0.0            0                     0   
3   8.3  97   4.0   0.2            0                     1   
4  11.4  99   1.8   0.0            0                     0   

   rain_evaporation_risk  
0                      0  
1                      0  
2                      0  
3                      0  
4                      0  


In [8]:
df_fe['area_log'] = np.log1p(df_fe['area'])
df_fe['daily_total_area_log'] = np.log1p(df_fe['daily_total_area'])

print("En Büyük 5 Yangın ve Logaritmik Dönüşümleri:")
print(df_fe[['area', 'area_log', 'daily_total_area', 'daily_total_area_log']].sort_values(by='area', ascending=False).head(5))

En Büyük 5 Yangın ve Logaritmik Dönüşümleri:
        area  area_log  daily_total_area  daily_total_area_log
238  1090.84  6.995620           1090.84              6.995620
415   746.28  6.616440            746.28              6.616440
479   278.53  5.633110            278.53              5.633110
237   212.88  5.365415            212.88              5.365415
236   200.94  5.307971            200.94              5.307971


In [9]:
def get_risk_class(area):
    if area <= 30.32:
        return 0  # Söndürülmüş veya Düşük Hasar (0 - 30.32 ha)
    elif area <= 105.66:
        return 1  # Orta Risk (30.32 - 105.66 ha)
    elif area <= 278.53:
        return 2  # Yüksek Risk (105.66 - 278.53 ha)
    else:
        return 3  # Mega-Yangın / Kritik Risk (> 278.53 ha)

df_fe['risk_class'] = df_fe['area'].apply(get_risk_class)

print("Sınıf 0 (Düşük):", len(df_fe[df_fe['risk_class']==0]))
print("Sınıf 1 (Orta):", len(df_fe[df_fe['risk_class']==1]))
print("Sınıf 2 (Yüksek):", len(df_fe[df_fe['risk_class']==2]))
print("Sınıf 3 (Mega-Yangın):", len(df_fe[df_fe['risk_class']==3]))

Sınıf 0 (Düşük): 476
Sınıf 1 (Orta): 32
Sınıf 2 (Yüksek): 7
Sınıf 3 (Mega-Yangın): 2


In [10]:
pd.set_option('display.max_columns', None)

print("--- FEATURE ENGINEERING SONRASI VERİ SETİ ---")
display(df_fe.head())

import os

save_dir = '../../data/processed'

if not os.path.exists(save_dir):
    os.makedirs(save_dir)

save_path = f'{save_dir}/processed_forestfires.csv'
df_fe.to_csv(save_path, index=False)

print(f"\nYeni veri setimiz '{save_path}' yoluna kaydedildi.")

--- FEATURE ENGINEERING SONRASI VERİ SETİ ---


,X,Y,month,day,FFMC,DMC,DC,ISI,temp,RH,wind,rain,area,month_num,day_num,same_condition_count,daily_total_area,ignition_type,distance_to_center,is_periphery,human_access_zone,dynamic_seasonal_hotspot_distance,day_type_Saturday,day_type_Sunday,day_type_Weekday,is_peak_season,month_sin,month_cos,day_sin,day_cos,moderate_wind_danger,has_rain,hot_and_dry,double_drought,ISI_x_DC,rain_evaporation_risk,area_log,daily_total_area_log,risk_class
0,7,5,mar,fri,86.2,26.2,94.3,5.1,8.2,51,6.7,0.0,0.0,3,5,1,0.0,0,3.162278,0,0,1.000000,0,0,1,1,1.000000,6.123234e-17,-9.749279e-01,-0.222521,0,0,0,0,480.93,0,0.0,0.0,0
1,7,4,oct,tue,90.6,35.4,669.1,6.7,18.0,33,0.9,0.0,0.0,10,2,1,0.0,0,3.000000,0,0,1.414214,0,0,1,0,-0.866025,5.000000e-01,9.749279e-01,-0.222521,0,0,0,1,4482.97,0,0.0,0.0,0
2,7,4,oct,sat,90.6,43.7,686.9,6.7,14.6,33,1.3,0.0,0.0,10,6,1,0.0,0,3.000000,0,0,1.414214,1,0,0,0,-0.866025,5.000000e-01,-7.818315e-01,0.623490,0,0,0,1,4602.23,0,0.0,0.0,0
3,8,6,mar,fri,91.7,33.3,77.5,9.0,8.3,97,4.0,0.2,0.0,3,5,1,0.0,0,4.472136,0,1,2.236068,0,0,1,1,1.000000,6.123234e-17,-9.749279e-01,-0.222521,1,1,0,0,697.50,0,0.0,0.0,0
4,8,6,mar,sun,89.3,51.3,102.2,9.6,11.4,99,1.8,0.0,0.0,3,7,1,0.0,0,4.472136,0,1,2.236068,0,1,0,1,1.000000,6.123234e-17,-2.449294e-16,1.000000,0,0,0,0,981.12,0,0.0,0.0,0



Yeni veri setimiz '../../data/processed/processed_forestfires.csv' yoluna kaydedildi.


In [13]:
df_fe.head(10)

,X,Y,month,day,FFMC,DMC,DC,ISI,temp,RH,wind,rain,area,month_num,day_num,same_condition_count,daily_total_area,ignition_type,distance_to_center,is_periphery,human_access_zone,dynamic_seasonal_hotspot_distance,day_type_Saturday,day_type_Sunday,day_type_Weekday,is_peak_season,month_sin,month_cos,day_sin,day_cos,moderate_wind_danger,has_rain,hot_and_dry,double_drought,ISI_x_DC,rain_evaporation_risk,area_log,daily_total_area_log,risk_class
0,7,5,mar,fri,86.2,26.2,94.3,5.1,8.2,51,6.7,0.0,0.0,3,5,1,0.0,0,3.162278,0,0,1.000000,0,0,1,1,1.000000,6.123234e-17,-9.749279e-01,-0.222521,0,0,0,0,480.93,0,0.0,0.0,0
1,7,4,oct,tue,90.6,35.4,669.1,6.7,18.0,33,0.9,0.0,0.0,10,2,1,0.0,0,3.000000,0,0,1.414214,0,0,1,0,-0.866025,5.000000e-01,9.749279e-01,-0.222521,0,0,0,1,4482.97,0,0.0,0.0,0
2,7,4,oct,sat,90.6,43.7,686.9,6.7,14.6,33,1.3,0.0,0.0,10,6,1,0.0,0,3.000000,0,0,1.414214,1,0,0,0,-0.866025,5.000000e-01,-7.818315e-01,0.623490,0,0,0,1,4602.23,0,0.0,0.0,0
3,8,6,mar,fri,91.7,33.3,77.5,9.0,8.3,97,4.0,0.2,0.0,3,5,1,0.0,0,4.472136,0,1,2.236068,0,0,1,1,1.000000,6.123234e-17,-9.749279e-01,-0.222521,1,1,0,0,697.50,0,0.0,0.0,0
4,8,6,mar,sun,89.3,51.3,102.2,9.6,11.4,99,1.8,0.0,0.0,3,7,1,0.0,0,4.472136,0,1,2.236068,0,1,0,1,1.000000,6.123234e-17,-2.449294e-16,1.000000,0,0,0,0,981.12,0,0.0,0.0,0
5,8,6,aug,sun,92.3,85.3,488.0,14.7,22.2,29,5.4,0.0,0.0,8,7,1,0.0,0,4.472136,0,1,0.000000,0,1,0,1,-0.866025,-5.000000e-01,-2.449294e-16,1.000000,1,0,1,0,7173.60,0,0.0,0.0,0
6,8,6,aug,mon,92.3,88.9,495.6,8.5,24.1,27,3.1,0.0,0.0,8,1,1,0.0,0,4.472136,0,1,0.000000,0,0,1,1,-0.866025,-5.000000e-01,7.818315e-01,0.623490,0,0,1,0,4212.60,0,0.0,0.0,0
7,8,6,aug,mon,91.5,145.4,608.2,10.7,8.0,86,2.2,0.0,0.0,8,1,1,0.0,0,4.472136,0,1,0.000000,0,0,1,1,-0.866025,-5.000000e-01,7.818315e-01,0.623490,0,0,0,1,6507.74,0,0.0,0.0,0
8,8,6,sep,tue,91.0,129.5,692.6,7.0,13.1,63,5.4,0.0,0.0,9,2,1,0.0,0,4.472136,0,1,0.000000,0,0,1,1,-1.000000,-1.836970e-16,9.749279e-01,-0.222521,1,0,0,1,4848.20,0,0.0,0.0,0
9,7,5,sep,sat,92.5,88.0,698.6,7.1,22.8,40,4.0,0.0,0.0,9,6,1,0.0,0,3.162278,0,0,1.414214,1,0,0,1,-1.000000,-1.836970e-16,-7.818315e-01,0.623490,1,0,1,1,4960.06,0,0.0,0.0,0
